Load the CMS catalog

In [0]:
import requests

catalog = requests.get("https://data.cms.gov/data.json", timeout=120).json()
print(catalog.keys())
print(len(catalog["dataset"]), "datasets in the catalog")

Find the Part D dataset

In [0]:
for section in ["dataset", "datasetSeries"]:
    for d in catalog.get(section, []):
        title = d.get("title", "")
        if "part d" in title.lower() and "prescriber" in title.lower():
            print(section, "|", repr(title))

In [0]:
partd_2021 = next(d for d in catalog["dataset"]
                  if d["title"] == "Medicare Part D Prescribers - by Provider : 2021-12-03")

for dist in partd_2021["distribution"]:
    print(dist.get("title"), "|", dist.get("mediaType"), "|", dist.get("downloadURL"))

Download all three years

In [0]:
import requests, os

prefix = "Medicare Part D Prescribers - by Provider : "

for year in [2019, 2020, 2021]:
    # find this year's entry (the dates in the titles vary, so match on the start + year)
    ds = next(d for d in catalog["dataset"]
              if d["title"].startswith(prefix) and str(year) in d["title"])

    # grab the CSV link
    url = next(dist["downloadURL"] for dist in ds["distribution"]
               if dist.get("mediaType") == "text/csv")

    dest = f"/Volumes/workspace/default/raw/partd_{year}.csv"
    if os.path.exists(dest):
        print(year, "already downloaded")
        continue

    print(year, "downloading", url)
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    print(year, f"saved {os.path.getsize(dest) / 1e6:.0f} MB")

Keep only NC and save it as a table

In [0]:
from pyspark.sql import functions as F

for year in [2019, 2020, 2021]:
    df = spark.read.csv(f"/Volumes/workspace/default/raw/partd_{year}.csv",
                        header=True, inferSchema=True)
    nc = df.filter(F.col("Prscrbr_State_Abrvtn") == "NC").withColumn("year", F.lit(year))
    nc.write.mode("overwrite").saveAsTable(f"workspace.default.partd_nc_{year}")
    print(year, nc.count(), "NC prescribers")

In [0]:
df = spark.table("workspace.default.partd_nc_2021").toPandas()
df.head()

Load all three years into one table

In [0]:
df = (spark.table("workspace.default.partd_nc_2019")
      .unionByName(spark.table("workspace.default.partd_nc_2020"), allowMissingColumns=True)
      .unionByName(spark.table("workspace.default.partd_nc_2021"), allowMissingColumns=True)
      .toPandas())
print(df.shape)
df.head()


Same doctors across years?

In [0]:
df.groupby("Prscrbr_NPI")["year"].nunique().value_counts()

Top specialties, by year

In [0]:
df.groupby("year")["Prscrbr_Type"].value_counts().groupby(level=0).head(5)

Is opioid prescribing dropping?

In [0]:
df.groupby("year")["Opioid_Prscrbr_Rate"].median()

Mostly-blank columns

In [0]:
df.isna().mean().sort_values(ascending=False).head(15)

How many tiny prescribers?

In [0]:
(df["Tot_Clms"] < 50).mean()

Opioid rate histogram for one specialty

In [0]:
import matplotlib.pyplot as plt

fp = df[df["Prscrbr_Type"] == "Family Practice"]
fp["Opioid_Prscrbr_Rate"].hist(bins=50)
plt.xlabel("Opioid prescribing rate (%)")
plt.ylabel("Number of prescribers")
plt.title("Family Practice: opioid prescribing rate, NC 2019-2021")
plt.show()

In [0]:
cols = ["Tot_Clms", "Tot_Drug_Cst", "Tot_Benes", "Tot_Day_Suply",
        "Brnd_Tot_Clms", "Opioid_Tot_Clms", "Opioid_Prscrbr_Rate",
        "Opioid_LA_Tot_Clms", "Antbtc_Tot_Clms", "Bene_Avg_Risk_Scre"]
df[cols].isna().mean().sort_values(ascending=False)

In [0]:
blank = df["Opioid_Tot_Clms"].isna()
print("Exactly 0 opioids:", (df["Opioid_Tot_Clms"] == 0).mean())
print("Blank opioids:", blank.mean())
print("Median total claims, blank-opioid prescribers:", df.loc[blank, "Tot_Clms"].median())
print("Median total claims, everyone:", df["Tot_Clms"].median())

see the tail more clearly

In [0]:
import numpy as np
import matplotlib.pyplot as plt

bins = np.arange(0, 20.5, 0.5)  # same half-point buckets for every year, 0 to 20%
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, year in zip(axes, [2019, 2020, 2021]):
    rates = fp.loc[fp["year"] == year, "Opioid_Prscrbr_Rate"]
    ax.hist(rates, bins=bins, color="steelblue")
    ax.axvline(rates.median(), color="red", linestyle="--",
               label=f"median {rates.median():.1f}%")
    ax.set_title(f"Family Practice, {year}")
    ax.set_xlabel("Opioid prescribing rate (%)")
    ax.legend()

axes[0].set_ylabel("Number of prescribers")
plt.tight_layout()
plt.show()

## CHARTS


In [0]:
import os
FIG = "/Volumes/workspace/default/raw/figures"
os.makedirs(FIG, exist_ok=True)
print("Saving charts to:", FIG)

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.edgecolor": "#8a8985", "axes.titleweight": "bold", "axes.titlesize": 13,
    "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
})
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
GRAY, INK = "#b5b4ae", "#0b0b0b"
YEARS = [2019, 2020, 2021]

###NPs and PAs drove the growth

In [0]:
top5 = df["Prscrbr_Type"].value_counts().head(5).index
counts = df[df["Prscrbr_Type"].isin(top5)].groupby(["Prscrbr_Type", "year"]).size().unstack().loc[top5]

fig, ax = plt.subplots(figsize=(8, 4.5))
for color, (spec, row) in zip([BLUE, ORANGE, AQUA, YELLOW, MAGENTA], counts.iterrows()):
    change = (row[2021] / row[2019] - 1) * 100
    ax.plot(YEARS, row.values, marker="o", lw=2, ms=8, color=color, label=f"{spec} ({change:+.0f}%)")
ax.set_xticks(YEARS)
ax.set_ylabel("Prescribers")
ax.set_title("NC prescribers by specialty, 2019 to 2021")
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

### How many years each prescriber appears

In [0]:
yrs = df.groupby("Prscrbr_NPI")["year"].nunique().value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(["1 year", "2 years", "3 years"], yrs.values, color=[GRAY, BLUE, BLUE], width=0.6)
for b, v in zip(bars, yrs.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v / yrs.sum():.0%})", ha="center", va="bottom", color=INK)
ax.set_ylim(0, yrs.max() * 1.2)
ax.set_ylabel("Unique prescribers")
ax.set_title("Years each prescriber appears (blue = kept)")
plt.tight_layout(); plt.show()

### The overall drop vs the family doctor drop

In [0]:
med_all = df.groupby("year")["Opioid_Prscrbr_Rate"].median()
med_fp = df[df["Prscrbr_Type"] == "Family Practice"].groupby("year")["Opioid_Prscrbr_Rate"].median()

fig, ax = plt.subplots(figsize=(8, 4.5))
for med, color, name in [(med_all, BLUE, "All prescribers"), (med_fp, ORANGE, "Family Practice")]:
    idx = med / med[2019] * 100
    ax.plot(YEARS, idx.values, marker="o", lw=2, ms=8, color=color, label=f"{name} ({idx[2021] - 100:+.0f}%)")
    for y in YEARS:
        offset = 9 if name == "Family Practice" else -15
        ax.annotate(f"{med[y]:.1f}%", (y, idx[y]), xytext=(0, offset), textcoords="offset points",
                    ha="center", fontsize=9, color=INK)
ax.set_xticks(YEARS)
ax.set_ylabel("Median opioid rate (2019 = 100)")
ax.set_title("Median opioid rate, indexed to 2019")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

###What a blank opioid count means

In [0]:
grp = np.select([df["Opioid_Tot_Clms"] == 0, df["Opioid_Tot_Clms"].isna()],
                ["0 opioids", "1 to 10 (hidden)"], "11+ (shown)")
share = pd.crosstab(df["year"], grp, normalize="index")[["0 opioids", "1 to 10 (hidden)", "11+ (shown)"]] * 100

fig, ax = plt.subplots(figsize=(8, 3.5))
left = np.zeros(len(share))
for col, color in zip(share.columns, [GRAY, ORANGE, BLUE]):
    ax.barh([str(y) for y in share.index], share[col], left=left, color=color, edgecolor="white", linewidth=2, label=col)
    for i, v in enumerate(share[col]):
        ax.text(left[i] + v / 2, i, f"{v:.0f}%", ha="center", va="center", color="white", fontweight="bold")
    left += share[col].values
ax.set_xlim(0, 100); ax.set_xlabel("Share of prescribers (%)")
ax.set_title("Opioid claims: zero, hidden, or shown")
ax.legend(frameon=False, ncol=3, bbox_to_anchor=(0.5, -0.25), loc="upper center")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

### Missingness in the columns you’ll use

In [0]:
names = {"Brnd_Tot_Clms": "Brand-name claims", "Opioid_LA_Tot_Clms": "Long-acting opioid claims",
         "Opioid_Tot_Clms": "Opioid claims", "Antbtc_Tot_Clms": "Antibiotic claims",
         "Tot_Benes": "Patients", "Bene_Avg_Risk_Scre": "Patient risk score",
         "Tot_Clms": "Total claims", "Tot_Drug_Cst": "Total drug cost", "Tot_Day_Suply": "Days supply"}
miss = (df[list(names)].isna().mean() * 100).rename(names).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(miss.index, miss.values, color=BLUE, height=0.6)
for i, v in enumerate(miss.values):
    ax.text(v + 0.8, i, f"{v:.0f}%", va="center", color=INK)
ax.set_xlim(0, miss.max() * 1.15); ax.set_xlabel("Blank (%)")
ax.set_title("Share blank in key columns, NC 2019 to 2021")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

### Why the 50-claim cutoff

In [0]:
hid = df[df["Opioid_Tot_Clms"].isna()].copy()
hid["low"] = 1 / hid["Tot_Clms"] * 100
hid["high"] = 10 / hid["Tot_Clms"] * 100
hid["bucket"] = pd.cut(hid["Tot_Clms"], [0, 49, 99, 249, 999, np.inf],
                       labels=["Under 50", "50-99", "100-249", "250-999", "1,000+"])
rng = hid.groupby("bucket", observed=True)[["low", "high"]].median()
n = hid.groupby("bucket", observed=True).size()

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(rng))
colors = [ORANGE] + [BLUE] * (len(rng) - 1)
for i, (lo, hi) in enumerate(rng.values):
    ax.plot([i, i], [lo, hi], color=colors[i], lw=10, solid_capstyle="round")
    ax.text(i + 0.15, hi, f"{lo:.1f}% to {hi:.1f}%", va="center", color=INK, fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels([f"{lab}\n({c:,} hidden)" for lab, c in zip(rng.index, n.values)])
ax.set_xlim(-0.5, len(rng) - 0.2)
ax.set_xlabel("Total claims per year")
ax.set_ylabel("Possible opioid rate (%)")
ax.set_title("Possible opioid rate range when the count is hidden (1 to 10), median prescriber")
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

### “Normal” depends on specialty

In [0]:
top8 = df["Prscrbr_Type"].value_counts().head(8).index
d = df[df["Prscrbr_Type"].isin(top8)]
order = d.groupby("Prscrbr_Type")["Opioid_Prscrbr_Rate"].median().sort_values().index
short = {"Student in an Organized Health Care Education/Training Program": "Residents/Trainees"}
data = [d.loc[d["Prscrbr_Type"] == s, "Opioid_Prscrbr_Rate"].dropna() for s in order]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(data, vert=False, widths=0.55,
           boxprops=dict(color=BLUE, lw=1.5), whiskerprops=dict(color=BLUE), capprops=dict(color=BLUE),
           medianprops=dict(color=ORANGE, lw=2.5),
           flierprops=dict(marker=".", markersize=3, alpha=0.3, markeredgecolor=GRAY, markerfacecolor=GRAY))
ax.set_yticks(range(1, len(order) + 1))
ax.set_yticklabels([short.get(s, s) for s in order])
ax.set_xlabel("Opioid prescribing rate (%)")
ax.set_title("Opioid rate by specialty (orange line = median)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### Sicker patients, pricier prescriptions

In [0]:
d = df[df["Tot_Clms"] >= 50].dropna(subset=["Bene_Avg_Risk_Scre", "Tot_Benes"]).copy()
d["rx_per_patient"] = d["Tot_Clms"] / d["Tot_Benes"]
d["cost_per_rx"] = d["Tot_Drug_Cst"] / d["Tot_Clms"]
d["risk_group"] = pd.cut(d["Bene_Avg_Risk_Scre"], [0, 1, 1.5, 2.5, np.inf],
                         labels=["Healthier\n(under 1)", "Average\n(1 to 1.5)",
                                 "Sicker\n(1.5 to 2.5)", "Much sicker\n(2.5+)"])
g = d.groupby("risk_group", observed=True)[["rx_per_patient", "cost_per_rx"]].median()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col, title, fmt, ylab in [
    (axes[0], "rx_per_patient", "Prescriptions per patient (median)", "{:.1f}", "Prescriptions per patient"),
    (axes[1], "cost_per_rx", "Cost per prescription (median)", "${:.0f}", "Cost per prescription ($)")]:
    bars = ax.bar(range(len(g)), g[col], color=BLUE, width=0.6)
    for b, v in zip(bars, g[col]):
        ax.text(b.get_x() + b.get_width() / 2, v, fmt.format(v), ha="center", va="bottom",
                color=INK, fontweight="bold")
    ax.set_xticks(range(len(g))); ax.set_xticklabels(g.index)
    ax.set_ylim(0, g[col].max() * 1.2)
    ax.set_title(title)
    ax.set_ylabel(ylab)
    ax.set_xlabel("Patient risk score group")
    ax.grid(axis="x", visible=False)
fig.suptitle("How sick the patients are vs what gets prescribed", fontweight="bold", fontsize=14)
plt.tight_layout(); plt.show()

### Rural vs urban

In [0]:
d = df.dropna(subset=["Prscrbr_RUCA"]).copy()
d["area"] = np.where(d["Prscrbr_RUCA"] < 4, "Metro (RUCA 1-3)", "Non-metro (RUCA 4-10)")
m = d.groupby(["year", "area"])["Opioid_Prscrbr_Rate"].median().unstack()

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(m.index)); w = 0.36
for i, (col, color) in enumerate(zip(m.columns, [BLUE, ORANGE])):
    bars = ax.bar(x + (i - 0.5) * (w + 0.02), m[col], width=w, color=color, label=col)
    for b, v in zip(bars, m[col]):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=9, color=INK)
ax.set_xticks(x); ax.set_xticklabels(m.index)
ax.set_ylim(0, m.values.max() * 1.2)
ax.set_ylabel("Median opioid rate (%)")
ax.set_title("Median opioid rate: metro vs non-metro")
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

print("Missing RUCA:", df["Prscrbr_RUCA"].isna().sum(), "rows")